# Hello, Endgame

A 5-minute tour of Endgame's core workflow: **train → compare → calibrate → report**.

We'll use the German Credit dataset (1000 samples, 20 features, binary classification) and keep everything under 30 lines of modeling code.

In [ ]:
import endgame as eg
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

## 1. Load data

In [ ]:
credit = fetch_openml(data_id=31, as_frame=True, parser="auto")
X, y = credit.data, (credit.target == "good").astype(int).values

# Encode categoricals
cat_cols = X.select_dtypes(include="category").columns
X = X.copy()
X[cat_cols] = OrdinalEncoder().fit_transform(X[cat_cols])
X = X.values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Train five models

Every Endgame model shares the scikit-learn `fit`/`predict` interface.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

models = {
    "LightGBM":     eg.models.LGBMWrapper(preset="endgame"),
    "EBM":          eg.models.EBMClassifier(),
    "RandomForest":  RandomForestClassifier(n_estimators=200, random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    acc = m.score(X_test, y_test)
    print(f"{name:15s}  accuracy={acc:.4f}")

## 3. Quick model comparison

`eg.quick.compare()` trains and cross-validates multiple models in one call.

In [ ]:
comparison = eg.quick.compare(X_train, y_train, task="classification", preset="default")
print(comparison)

## 4. Calibrate with conformal prediction

Wrap the best model in a `ConformalClassifier` to get prediction sets with coverage guarantees.

In [ ]:
from endgame.calibration import ConformalClassifier

best = models["LightGBM"]

# Split test into calibration + final test
X_cal, X_final, y_cal, y_final = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

conf = ConformalClassifier(best, method="lac", alpha=0.05)
conf.fit(X_train, y_train, X_cal, y_cal)

coverage = conf.coverage_score(X_final, y_final)
print(f"Empirical coverage: {coverage:.1%} (target: 95%)")

## 5. Generate an interactive HTML report

Self-contained HTML with confusion matrix, ROC curve, calibration plot, feature importances, and more.

In [ ]:
from endgame.visualization import ClassificationReport

report = ClassificationReport(
    best, X_test, y_test,
    feature_names=credit.feature_names,
    model_name="LightGBM (endgame preset)",
    dataset_name="German Credit",
)
report.save("hello_endgame_report.html")
print("Report saved to hello_endgame_report.html")

## Next steps

- **`01_quickstart.ipynb`** — Full walkthrough of training, evaluation, and persistence
- **`02_interpretable_models.ipynb`** — 30+ glass-box models for regulated domains
- **`03_automl.ipynb`** — AutoML with guardrails and deployment constraints
- **`04_ensemble_calibration.ipynb`** — Ensemble optimization and calibration

See [ROADMAP.md](../ROADMAP.md) for the full feature list.